In [ ]:
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice



    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    def read_table_context(text_path: str):

        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        second_line = (second_line or "").strip()
        if " " in second_line:
            second_line_clean = second_line.split(" ", 1)[1]
        else:
            second_line_clean = ""  

        return final_title, second_line_clean
    

    def read_table_context1(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            tail_parts.append(s.strip())

        if tail_parts:
            if second_line_clean:
                second_line_clean = second_line_clean + " " + " ".join(tail_parts)
            else:
                second_line_clean = " ".join(tail_parts)

        return final_title, second_line_clean

    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        # чисто "1 2 3" / "5"
        if ONLY_DIGITS_SPACES.match(s):
            return True
        # если строка вообще без букв и состоит из цифр+разделителей
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context2(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean



    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    def merge_parts(df):
        df = df.copy()

        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }

        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered

        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        result = (
            df.groupby("Part Number", as_index=False)
            .agg(agg_map)
        )
        return result

    for i, pth in enumerate(tqdm(pdf_files[1:2])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context2(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )

        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged['Model'] = 'AMWP11.5-8200AC'
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)


  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA16  18NE  Parts Manual （SM042220111_Rev1.1）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1.0,NaN,Left Hub Assembly (Refer to 1-2),1,NaN,Chassis,Steer Cylinder Assembly Installations
1,2.0,50000921.0,"Pin, Pivot",4,NaN,Chassis,Steer Cylinder Assembly Installations
2,3.0,50000926.0,Tie-Rod,2,NaN,Chassis,Steer Cylinder Assembly Installations
3,4.0,1257.0,Nut GB/T 6182 M16,4,NaN,Chassis,Steer Cylinder Assembly Installations
4,5.0,997.0,Flat Washer GB/T 97.1 16,4,NaN,Chassis,Steer Cylinder Assembly Installations


100%|██████████| 1/1 [00:02<00:00,  2.36s/it]


плохие без картинок: 3:4, 5:6, 6:7, 7:8, 8:9, 9:10
хорошие с картинками: 10:11, 11:12, 15:16, 17:18
норм: 0:1, 1:2, 2:3, 4:5 
плохие с картинками: 
done: 12:13(r"C:\Users\Normc\OneDrive\Рабочий стол\Python\Работа\Dingli\JCPT0808-1612AC HA Parts Manual（SM012020111_Rev1.6）.pdf")
done 13:14 (r"C:\Users\Normc\OneDrive\Рабочий стол\Python\Работа\Dingli\JCPT0808-1612HA AC Parts Manual（SM012020111_Rev1.6）.pdf") 
done 14:15 (r"C:\Users\Normc\OneDrive\Рабочий стол\Python\Работа\Dingli\JCPT0808-1614 Магни  备件手册  英文（SM0117217_Rev1.0）.pdf")
done 16:17 (r"C:\Users\Normc\OneDrive\Рабочий стол\Python\Работа\Dingli\JCPT1523RTB 1823RTB-SM012120119_Rev1.1-Parts Manual.pdf")
done 18:19 (r"C:\Users\Normc\OneDrive\Рабочий стол\Python\Работа\Dingli\JCPT1912DC  2212DC.pdf")

In [ ]:
0:1: df_merged['Model'] = 'AMWP11.5-8200AC'
1:2: 

In [67]:
import pdfplumber

pdf_path = pth

with pdfplumber.open(pdf_path) as pdf:
    first_page = pdf.pages[0]          # первая страница (индексация с 0)
    text = first_page.extract_text()   # текст со страницы
    print(text or "")

SELF-PROPELLED SCISSOR LIFTS
PARTS MANUAL
( For JCPT0708DCS )
WAR NING
THE MANUFACTURER SHALL NOT BE HELD LIABLE IN CASE OF FAULTS
OR ACCIDENTS DUE TO NEGLIGENCE, INCAPACITY, INSTALLATION BY
UNQUALIFIED TECHNICIANS AND IMPROPER USE OF THE MACHINE
DO NOT OPERATE THIS MACHINE UNTIL YOU READ AND UNDERSTAND
ALL THE DANGERS,WARNINGS AND CAUTIONS IN THIS MANUAL
Part Number: SM0117225_Rev2.4 ANSI
Zhejiang Dingli Machinery Co., Ltd. Second Edition, November 2019 Printing


Part Number:, From SN; WAR; WARNING

In [ ]:
def extract_tables_camelot(pdf_path: str, out_dir="tables"):
    os.makedirs(out_dir, exist_ok=True)

    tables_lattice = camelot.read_pdf(pdf_path, pages="1-20", flavor="lattice")

    for i, t in enumerate(tables_lattice):
        t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"))

    return tables_lattice


extract_tables_camelot(pth, filename);

In [ ]:
dfs = []

for file in os.listdir(filename):
    if file.startswith("lattice_"):
        path = os.path.join(filename, file)
        df_i = pd.read_csv(path)
        if 'Item' not in df_i:
            continue
        if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
            df_i.columns.values[1] = "Part Number"
            df_i.columns.values[2] = "Description"
            df_i.columns.values[3] = "Qty."
        if 'Note' in df_i.columns:
            df_i.columns.values[4] = "Remark"
            
        dfs.append(df_i)

df_all = pd.concat(dfs, ignore_index=True)
cols = df_all.columns.tolist()

if 'Part Number' not in cols:
    cols[1] = "Part Number"
    df_all.columns = cols


display(df_all.iloc[:, 1].isna().sum())
display(df_all)
df_all = df_all.dropna(subset=[df_all.columns[1]])

In [ ]:
# col = df_all.iloc[:, 1].astype(str)

# col_numeric = pd.to_numeric(col, errors='coerce')
# df_letters = df_all[col_numeric.isna()].copy()

In [ ]:
# df_letters

# df_numbers = df_all[col_numeric.notna()].copy()
# df_numbers.iloc[:, 1] = col_numeric[col_numeric.notna()].astype(int)

In [ ]:
df_all = df_all[~(df_all['Qty.'] == '/')]
df_all = df_all[~(df_all['Qty.'].isna())]

In [ ]:
mask = df_all.iloc[:,1].apply(lambda x: not str(x).replace('.0','').isdigit())
df_all[mask]
display(df_all)
df_all['Qty'] = df_all['Qty.']
df_all = df_all.drop(columns=['Qty.', 'Item'])
df_all['Qty'] = df_all['Qty'].astype(int)

if 'Remark' in df_all.columns:
    df_all["Remark"] = df_all["Remark"].str.replace("\n", " ", regex=False)
    df_all["Remark"] = df_all["Remark"].str.strip().str.replace(r"\s+", " ", regex=True)

In [ ]:
def join_unique(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v and v.lower() != "nan"]
    return "; ".join(set(vals))


def join_uniques(series):
    vals = series.dropna().astype(str).str.strip()
    uniq = []
    for v in vals:
        for item in v.split(): 
            if item not in uniq or item == 'Olny':
                uniq.append(item)
    return " ".join(uniq) if uniq else None


def merge_parts(df):
    df = df.copy()
    if 'Remark' in df.columns:
        result = (
            df.groupby("Part Number", as_index=False)
            .agg({
                "Description": join_unique,
                "Qty": "sum",
                "Remark": join_uniques
            })
        )
    else:
        result = (
        df.groupby("Part Number", as_index=False)
        .agg({
            "Description": join_unique,
            "Qty": "sum",
        })
    )

    return result

df_merged = merge_parts(df_all)

df_merged['Part Number'] = df_merged['Part Number'].apply(lambda x: f"{int(x):08d}") 



In [ ]:
df_letters['Qty'] = df_letters['Qty.']
df_letters = df_letters.drop(columns=['Qty.', 'Item'])

if 'Remark' in df_letters.columns:
    df_letters["Remark"] = df_letters["Remark"].str.replace("\n", " ", regex=False)
    df_letters["Remark"] = df_letters["Remark"].str.strip().str.replace(r"\s+", " ", regex=True)

In [ ]:
df_merged = pd.concat([df_letters, df_merged], ignore_index=True)


In [ ]:
df_merged.to_excel(rf"{filename}.xlsx", sheet_name="Parts", index=False)